In [1]:
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import re

In [2]:
import sys, os

# AutomationModule lives in the root of the tidy3d project
sys.path.append(os.path.abspath(r'H:\codes\tidy3d'))
import AutomationModule as AM


In [3]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [4]:
lambdas = np.array([3.5,1.5])
run = True


params example:{'aspect_ratio': 1,
 'background_permittivity': 1.0,
 'box_size': array([12.        , 12.        ,  8.48528137]),
 'd': 1.2,
 'defect_density': 0.09984609641754491,
 'dz': 1.697056274847714,
 'ff': 0.450616519434629,
 'ff_analytic': 0.4101477321755692,
 'grid_size': array([400, 400, 283]),
 'kappa': 2.8,
 'major_radius': 0.25781249999999994,
 'minor_radius': 0.25781249999999994,
 'permittivity': 6.25,
 'seed': 12345}

In [5]:
project_name = "20260918_Transmission_Experiment_woodpiles"
folder_path = rf"./Structures"
direction = "z" 
runtime_ps = 22e-12
min_steps_per_lambda = 18
for dirpath, dirnames, filenames in os.walk(folder_path):
    for filename in filenames:
        ref = True
        if filename.endswith(".h5"):
            print(f"Processing file: {filename}")
            if os.path.isfile(os.path.join(dirpath, filename)):
                file=os.path.join(dirpath, filename)
                params = AM.read_hdf5_as_dict(file)["params"]
                structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file
                                                     ,direction=direction, lambda_range=lambdas,
                                                     box_size=list(params["box_size"]),runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                                                    scaling=1,shuoff_condtion=1e-20, verbose=True, 
                                                    monitors=["flux"],
                                                    freqs=300, 
                                                     source="planewave", absorbers=130,sim_name=rf"{Path(filename).stem}"
                                                    )
                if run:
                    id0=""
                    sim=structure_1.sim
                    folder_desc = rf"{Path.cwd().parents[1]}/data/{project_name}/n_{np.sqrt(params['permittivity']):.2f}"
                    os.makedirs(folder_desc, exist_ok=True)
                    sim_name=rf"{Path(filename).stem}"
                    if os.path.exists(os.path.join(folder_desc, sim_name+".txt")):
                        print(os.path.join(folder_desc, sim_name+".txt"))
                        print("Exist!")
                        continue
                    else:
                        if ref:
                            sim0=sim.copy(update={"structures":[]})
                            id0 =web.upload(sim0, folder_name=project_name,task_name=sim_name+"_0", verbose=True)
                            web.start(task_id = id0)
                            web.monitor(id0)
                            ref=False

                        id =web.upload(sim, folder_name=project_name,task_name=sim_name, verbose=True)
                        ids = id0+'\n' + id
                        with open(os.path.join(folder_desc, sim_name+".txt"), "w") as file:
                            # Write the string to the file
                            file.write(ids)
                        web.start(task_id = id)
                        web.monitor(id)
                else: 
                    structure_1.plot_sim_layout()

       

Processing file: n_2.50_ff_0.3996_woodpile_d1.20_kappa+0.00_rho0.000_seednone_tables.h5
Configured successfully.
h:\Codes\tidy3d/data/20260918_Transmission_Experiment_woodpiles/n_2.50\n_2.50_ff_0.3996_woodpile_d1.20_kappa+0.00_rho0.000_seednone_tables.txt
Exist!
Processing file: n_2.50_ff_0.4229_woodpile_d1.20_kappa+1.20_rho0.100_seed12345_tables.h5
Configured successfully.


14:48:15 W. Europe Daylight Time Created task                                   
                                 'n_2.50_ff_0.4229_woodpile_d1.20_kappa+1.20_rho
                                 0.100_seed12345_tables_0' with task_id         
                                 'fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3' and
                                 task_type 'FDTD'.

                                 View task using web UI at                      
                                 ]8;id=897751;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=832535;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\task]8;;\
                                 ]8;id=832535;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\Id]8;;\]8;id=897751;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\=]8;;\]8;id=546525;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\fdve]8;;\]8;id=897751;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0fb71201-de56-4cc1-9394-39465dd0fbf3\-0fb71201-de56-4cc1-9394-39465dd0fbf3']8;;\.

                                 Task folder:                                   
                                 ]8;id=778629;https://tidy3d.simulation.cloud/folders/folder-f1fd42c7-e513-4349-8d1f-d6f336e103ee\'20260918_Transmission_Experiment_woodpiles']8;;\.

Output()

...

14:52:40 W. Europe Daylight Time status = postprocess

Output()

14:52:45 W. Europe Daylight Time status = success

14:52:47 W. Europe Daylight Time View simulation result at                      
                                 ]8;id=475306;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=667652;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\task]8;;\
                                 ]8;id=667652;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\Id]8;;\]8;id=475306;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\=]8;;\]8;id=892121;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\fdve]8;;\]8;id=475306;https://tidy3d.simulation.cloud/workbench?taskId=fdve-6cbec793-6d92-44e7-9248-93fab382e67f\-6cbec793-6d92-44e7-9248-93fab382e67f']8;;\.

Processing file: n_2.50_ff_0.4300_woodpile_d1.20_kappa+1.60_rho0.100_seed12345_tables.h5
Configured successfully.
h:\Codes\tidy3d/data/20260918_Transmission_Experiment_woodpiles/n_2.50\n_2.50_ff_0.4300_woodpile_d1.20_kappa+1.60_rho0.100_seed12345_tables.txt
Exist!
Processing file: n_2.50_ff_0.4506_woodpile_d1.20_kappa+2.80_rho0.100_seed12345_tables.h5
Configured successfully.
h:\Codes\tidy3d/data/20260918_Transmission_Experiment_woodpiles/n_2.50\n_2.50_ff_0.4506_woodpile_d1.20_kappa+2.80_rho0.100_seed12345_tables.txt
Exist!
